# Step 3 — Curate in FiftyOne and export to YOLO format

The VLM gave us bounding boxes in notebook 02. Before we train on them, a human should look at the results. FiftyOne's app lets us scroll through predictions and reject obviously-wrong samples with a click.

At the end of this notebook, `datasets/vlm-annotated/` contains YOLO-format labels on disk that [`04-train-yolo.ipynb`](04-train-yolo.ipynb) will train on.

In [ ]:
%%capture
!pip install fiftyone

In [ ]:
import shutil
from pathlib import Path

import fiftyone as fo

CLASSES = ["person", "helmet", "vest", "gloves"]
EXPORT_ROOT = Path("datasets/vlm-annotated")
VLM_TAG = "vlm_annotated"

## Open the FiftyOne app for a human QA pass

Scroll through the `vlm_annotated` view. If you spot a bad prediction, tag the sample `reject` in the app's sample sidebar — we'll filter those out before exporting.

> On a RHOAI workbench, use the proxy URL pattern from notebook 01 instead of `launch_app`.

In [ ]:
dataset = fo.load_dataset("ppe")
annotated_view = dataset.match_tags(VLM_TAG)
print(f"{len(annotated_view)} samples with VLM predictions.")

fo.close_app()
session = fo.launch_app(annotated_view, port=5151, address="0.0.0.0")
session.show()

## Drop rejected samples

After the QA pass, keep everything that wasn't tagged `reject`.

In [ ]:
from fiftyone import ViewField as F

# Keep VLM-annotated samples that the QA pass didn't tag "reject".
keep = dataset.match_tags(VLM_TAG).match(~F("tags").contains("reject"))
print(f"After curation: {len(keep)} samples.")

## Export to YOLO v5 format

FiftyOne writes an Ultralytics-compatible layout under `datasets/vlm-annotated/`:
```
images/train/*.jpg        # the 40 VLM-annotated images
labels/train/*.txt        # one YOLO-format .txt per image
dataset.yaml              # FiftyOne's generated yaml
```

Ultralytics needs both a `train` and a `val` split. We reuse the same 40 images for both — with only 40 samples, holding some out for val produces noisier numbers than just measuring train-fit. That's an honest workshop simplification; call it out if you're showing this to an ML audience.

In [ ]:
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

keep.export(
    export_dir=str(EXPORT_ROOT),
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="vlm_predictions",
    classes=CLASSES,
    split="train",
)

# Overwrite FiftyOne's dataset.yaml to point val at the same train images.
yaml_body = (
    "path: .\n"
    "train: images/train\n"
    "val: images/train\n"
    "names:\n" + "".join(f"  {i}: {c}\n" for i, c in enumerate(CLASSES))
)
(EXPORT_ROOT / "dataset.yaml").write_text(yaml_body)

print("Exported to:", EXPORT_ROOT.resolve())

In [ ]:
!ls {EXPORT_ROOT}
!echo '--- dataset.yaml ---'
!cat {EXPORT_ROOT}/dataset.yaml
!echo '--- sample label file ---'
!ls {EXPORT_ROOT}/labels/train | head -1 | xargs -I{{}} cat {EXPORT_ROOT}/labels/train/{{}}

---

The `datasets/vlm-annotated/dataset.yaml` FiftyOne just wrote is what [`04-train-yolo.ipynb`](04-train-yolo.ipynb) feeds into `yolo train`. The project-root [`data-vlm.yaml`](data-vlm.yaml) is a simplified copy kept under version control as documentation of the 4-class taxonomy.